In [1]:
#Checking PyTorch and GPU
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [2]:
#Getting GPU Information
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

GPU: Tesla T4


In [3]:
#Installing Required Libraries

!pip install torch torchvision torchaudio transformers datasets accelerate

In [2]:
from transformers import pipeline

In [1]:
#Summarization

In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

text = "Artificial intelligence is transforming the world."

inputs = tokenizer(text, return_tensors="pt")

summary_ids = model.generate(inputs["input_ids"], max_length=40)

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

d:\New folder\Samir\Lib\site-packages\transformers\generation\utils.py:1575: UserWarning: Unfeasible length constraints: `min_length` (56) is larger than the maximum possible length (40). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length.
  warnings.warn(


Artificial intelligence is transforming the world. Here are some of the ways in which it is changing our lives. And how it's changing the way we see the world around us too.


In [6]:
!pip install transformers datasets torch gradio accelerate

In [4]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [5]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})


In [6]:
print(dataset["train"][0]["article"])
print(dataset["train"][0]["highlights"])

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office chart. Details of how

In [7]:
model_name = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

In [8]:
def preprocess(example):

    inputs = tokenizer(
        example["article"],
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        example["highlights"],
        max_length=128,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]

    return inputs

In [9]:
tokenized_dataset = dataset["train"].select(range(2000)).map(preprocess)

In [10]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    num_train_epochs=1,
    logging_steps=50,
    save_steps=500,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

d:\New folder\Samir\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,1.850910
100,1.777648
150,1.720752
200,1.700671
250,1.648282
300,1.740060
350,1.645715
400,1.597145
450,1.577685
500,1.522629


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=1.5965885467529297, metrics={'train_runtime': 32332.4637, 'train_samples_per_second': 0.062, 'train_steps_per_second': 0.031, 'total_flos': 2131148442746880.0, 'train_loss': 1.5965885467529297, 'epoch': 1.0})

In [11]:
!pip install evaluate
!pip install rouge_score


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=25027 sha256=2439cededaae2d35c5e72807cc78708288071d3c291c877baabf5ace0d8ff4ec
  Stored in directory: c:\users\asuss\appdata\local\pip\cache\wheels\44\af\da\5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score

   ---------------------------------------- 2/2 [rouge_score]



In [12]:
import evaluate

rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=["generated summary"],
    references=["actual summary"]
)

print(results)

{'rouge1': np.float64(0.5), 'rouge2': np.float64(0.0), 'rougeL': np.float64(0.5), 'rougeLsum': np.float64(0.5)}


In [13]:
def summarize(text):

    inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)

    # Move the input tensors to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=120,
        min_length=30
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    return summary

In [13]:
text = "Based on its paper, GPT-3 is an autoregressive language model as opposed to a denoising autoencoder like BERT. I decided to write about some of the comparative differences between those two architectures of language models. The paper is an investigation into what you can do with giant language models. Now this language model is an order of magnitude larger than anyone has ever built a language model."

print(summarize(text))

GPT-3 is an autoregressive language model as opposed to a denoising autoencoder like BERT .
The paper is an investigation into what you can do with giant language models .


In [14]:
model.save_pretrained("./fine_tuned_bart_model")
tokenizer.save_pretrained("./fine_tuned_bart_model")
print("Model and tokenizer saved to ./fine_tuned_bart_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved to ./fine_tuned_bart_model
